In [ ]:
!python3 -m pip --version
!python3 -m pip install "qiskit~=2.2" "qiskit-aer~=0.17" "qiskit-nature~=0.7" scipy
!pip install openfermion
!pip install --upgrade cirq
!pip install --quiet ply
!pip install qiskit_qasm3_import
!pip install qiskit

pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 73.6 

In [ ]:
### Testing https://github.com/vili-1/SimSHADOW_artifact/blob/main/simshadow/platforms/qiskit_platform.py

In [ ]:
"""
Qiskit platform implementation for SimSHADOW quantum simulator validation.

Provides Qiskit-specific quantum circuit execution and noise model integration.
"""
import numpy as np
from typing import List, Dict, Optional
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, phase_damping_error

import sys
from pathlib import Path

print(">> OK!")
PROJECT_ROOT = Path("/content/SimSHADOW_artifact-main/simshadow/")
sys.path.insert(0, str(PROJECT_ROOT))

sys.path.append("/content/SimSHADOW_artifact-main/simshadow/")

from core.shadow_tomography import QuantumState, PauliObservable
from core.noise_models import NoiseChannel, DepolarizingChannel, AmplitudeDampingChannel, PhaseDampingChannel
from core.shadow_tomography import create_pauli_observables


class QiskitPlatform:
    """
    Qiskit simulator platform for SimSHADOW experiments.
    Handles circuit construction, noise model configuration, and measurement execution
    specifically for Qiskit-based simulations.
    """

    def __init__(self, n_qubits: int = 2):
        self.platform_name = "Qiskit"
        self.n_qubits = n_qubits
        self.simulator = AerSimulator()
        self.noise_model: Optional[NoiseModel] = None
        self.current_noise_config: Optional[Dict] = None

    def configure_noise(self, noise_channel: NoiseChannel):
        """
        Configure Qiskit noise model based on SimSHADOW noise channel.
        Called from run_simshadow.py
        """
        self.current_noise_config = {
            'type': noise_channel.name,
            'parameter': noise_channel.parameter
        }

        # Create Qiskit noise model
        noise_model = NoiseModel()

        if isinstance(noise_channel, DepolarizingChannel):
            # Add depolarising errors to all single and two-qubit gates
            error_1q = depolarizing_error(noise_channel.p, 1)
            error_2q = depolarizing_error(noise_channel.p, 2)

            # Apply to common gates
            noise_model.add_all_qubit_quantum_error(error_1q, ['id', 'u1', 'u2', 'u3', 'h', 'x', 'y', 'z'])
            noise_model.add_all_qubit_quantum_error(error_2q, ['cx', 'cy', 'cz'])

        elif isinstance(noise_channel, AmplitudeDampingChannel):
            # Add amplitude damping errors
            error_1q = amplitude_damping_error(noise_channel.gamma)

            # Apply to all single-qubit gates
            noise_model.add_all_qubit_quantum_error(error_1q, ['id', 'u1', 'u2', 'u3', 'h', 'x', 'y', 'z'])

        elif isinstance(noise_channel, PhaseDampingChannel):
            # Add phase damping errors
            error_1q = phase_damping_error(noise_channel.lambda_param)

            # Apply to all single-qubit gates
            noise_model.add_all_qubit_quantum_error(error_1q, ['id', 'u1', 'u2', 'u3', 'h', 'x', 'y', 'z'])

        else:
            raise ValueError(
                "NoiseModel.name must be valid. ",
                noise_channel.name,
                "Use DepolarizingChannel, AmplitudeDampingChannel, or PhaseDampingChannel"
            )

        self.noise_model = noise_model

    def prepare_state_circuit(self, quantum_state: QuantumState) -> QuantumCircuit:
        """
        Create Qiskit circuit to prepare the specified quantum state.
        Internal function only.
        """
        # Here we will store the state!
        circuit = QuantumCircuit(self.n_qubits)

        if "GHZ" in quantum_state.name or "Φ" in quantum_state.name:
            # GHZ state preparation
            circuit.h(0)
            for i in range(1, self.n_qubits):
                circuit.cx(0, i)
        else: # We only have 0,1,+ and -

            # Explicitly keep only valid single-qubit symbols
            bitstring = [ch for ch in quantum_state.name if ch in {'0', '1', '+', '-'}]
            if len(bitstring) != self.n_qubits:
                raise ValueError(
                    f"State {quantum_state.name} does not match n_qubits={self.n_qubits}"
                )

            for i, ch in enumerate(bitstring):
                if ch == '0':
                    pass
                elif ch == '1':
                    circuit.x(i)
                elif ch == '+':
                    circuit.h(i)
                elif ch == '-':
                    circuit.h(i)
                    circuit.z(i)
                else:
                    raise ValueError(
                        "QuantumState.name must be valid. "
                        "Use 0, 1, +, - or GHZ only."
                    )
        return circuit

    def measure_pauli(self, quantum_state: QuantumState, pauli_string: str) -> str:
        """
        Measure quantum state in specified Pauli basis and return outcome.

        Args:
            quantum_state: The quantum state to measure
            pauli_string: Pauli string like 'XY', 'ZZ', etc.

        Returns:
            Measurement outcome as bitstring

        Called:
            measure_pauli from shadow_tomography.py
        """
        # Prepare initial state
        circuit = self.prepare_state_circuit(quantum_state)

        # Add Pauli basis rotations
        for i, pauli in enumerate(pauli_string):
            if pauli == 'X':
                circuit.ry(-np.pi/2, i)  # Rotate Y by -π/2 to measure X
            elif pauli == 'Y':
                circuit.rx(np.pi/2, i)   # Rotate X by π/2 to measure Y
            # Leave this to the full paper, with a proper theorem.
            #elif pauli == 'Z' or pauli == 'I':
            #    circuit.id(i)
            # Z measurement is in computational basis (no rotation needed)
            # Z and I: no rotation

        # Execute circuit with noise
        circuit.measure_all()
        if self.noise_model:
            job = self.simulator.run(
                transpile(circuit, self.simulator,optimization_level=0),
                shots=1, # To follow classical shadow, and this is the only place where shots are 1.
                noise_model=self.noise_model
            )
        else:
            job = self.simulator.run(
                transpile(circuit, self.simulator,optimization_level=0),
                shots=1 # To follow classical shadow, and this is the only place where shots are 1.
            )

        result = job.result()
        counts = result.get_counts()
        bitstring = next(iter(counts))
        outcome = bitstring[::-1] # Assume measurements applied to all qubits
        return outcome

    def compute_expectation_value(self,
                                quantum_state: QuantumState,
                                observable: PauliObservable,
                                shots: int = 10000) -> float:
        """
        Compute expectation value of Pauli observable with proper noise simulation.

        Args:
            quantum_state: The quantum state
            observable: Pauli observable to measure
            shots: Number of measurement shots

        Returns:
            Estimated expectation value

        Called:
            main from run_simshadow.py
        """
        # Prepare state circuit
        circuit = self.prepare_state_circuit(quantum_state)

        # Add Pauli basis rotations for measurement
        for i, pauli in enumerate(observable.pauli_string):
            if pauli == 'X':
                circuit.ry(-np.pi/2, i)
            elif pauli == 'Y':
                circuit.rx(np.pi/2, i)
            # Leave this to the full paper, with a proper theorem.
            #elif pauli == 'Z' or pauli == 'I':
            #    circuit.id(i)

        # Add measurements
        circuit.measure_all()

        # Execute with noise model
        if self.noise_model:
            job = self.simulator.run(
                transpile(circuit, self.simulator,optimization_level=0),
                shots=shots,
                noise_model=self.noise_model
            )
        else:
            job = self.simulator.run(
                transpile(circuit, self.simulator,optimization_level=0),
                shots=shots
            )

        result = job.result()
        counts = result.get_counts()

        # Compute expectation value from measurement statistics
        # For Pauli observables, eigenvalue is product of individual qubit eigenvalues
        # After rotating to measurement basis: |0⟩ → +1, |1⟩ → -1
        # For identity (I): eigenvalue is always +1
        expectation = 0.0
        total_shots = sum(counts.values())

        for bitstring, count in counts.items():
            # Compute eigenvalue as product of individual qubit eigenvalues
            eigenvalue = 1.0
            for i, pauli in enumerate(observable.pauli_string):
                if pauli == 'I':
                    # Identity always has eigenvalue +1
                    continue
                else:
                    # After rotation to Pauli basis, |0⟩ → +1, |1⟩ → -1
                    bit = int(bitstring[-1 - i])
                    qubit_eigenvalue = 1.0 if bit == 0 else -1.0
                    eigenvalue *= qubit_eigenvalue

            expectation += eigenvalue * count / total_shots

        return expectation

    ## WORKING VERSION OF IDEAL
    def get_ideal_expectation(self, quantum_state: QuantumState,
                              observable: PauliObservable,
                              shots: int = 10000) -> float:
        """Get ideal (noiseless) expectation value."""
        # Temporarily disable noise
        original_noise = self.noise_model
        self.noise_model = None

        expectation = self.compute_expectation_value(quantum_state, observable, shots)

        # Restore noise model
        self.noise_model = original_noise

        return expectation

print("==> Start.")

from collections import Counter

# So this works
print("Create 0:")
platform = QiskitPlatform(1)
state = QuantumState.computational_basis_state("0")
print(state)
print(state.to_string())

samples = []
for _ in range(10):
    outcome = platform.measure_pauli(state, "Z")
    samples.append(outcome)
print(Counter(samples))

print("Create 1:")
platform = QiskitPlatform(1)
state = QuantumState.computational_basis_state("1")
print(state)
print(state.to_string())

samples = []
for _ in range(10):
    outcome = platform.measure_pauli(state, "Z")
    samples.append(outcome)
print(Counter(samples))

# Set up platform with NO noise
print("=========================================== 1 Qubits ===========================")
platform = QiskitPlatform(1)
state = QuantumState.computational_basis_state("1")
print("The state: ", state.to_string())
print("==================== Idel:")
for obs in ["X", "Y", "Z"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f}")
print("==================== Noisy")
for obs in ["X", "Y", "Z"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")

print("=========================================== 2 Qubits ===========================")
platform = QiskitPlatform(2)
platform.configure_noise(DepolarizingChannel(0.01))

#state = state = QuantumState.ghz_state(2) #QuantumState.computational_basis_state("00")
superposition_patterns = ['+0', '-0', '0+', '0-']
pattern = superposition_patterns[2]
state = QuantumState.superposition_state(list(pattern))

print("The state: ", state.to_string())
print("==================== Ideal:")
for obs in ["XX","XY","XZ","YX","YY","YZ","ZX","ZY","ZZ"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    ideal_expectation = PauliObservable(obs).expectation_value(state)
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f} or {ideal_expectation:.4f} ")

print("==================== Noisy")
for obs in ["XX","XY","XZ","YX","YY","YZ","ZX","ZY","ZZ"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")


print("=========================================== 3 Qubits ===========================")
platform = QiskitPlatform(3)
platform.configure_noise(DepolarizingChannel(0.01))

state = QuantumState.ghz_state(3) #QuantumState.computational_basis_state("00")
print(state)
print("The state: ", state.to_string())
print("==================== Ideal:")
for obs in ["XXX","XXY","XXZ","XYX","XYY","XYZ","XZX","XZY","XZZ","YXX","YXY","YXZ","YYX","YYY","YYZ","YZX","YZY","YZZ","ZXX","ZXY","ZXZ","ZYX","ZYY","ZYZ","ZZX","ZZY","ZZZ"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f}")

print("==================== Noisy")
for obs in ["XXX","XXY","XXZ","XYX","XYY","XYZ","XZX","XZY","XZZ","YXX","YXY","YXZ","YYX","YYY","YYZ","YZX","YZY","YZZ","ZXX","ZXY","ZXZ","ZYX","ZYY","ZYZ","ZZX","ZZY","ZZZ"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")

for ind in [1, 2, 3]:
    print("=========================================== ", ind, " Qubits ===========================")
    platform = QiskitPlatform(ind)
    platform.configure_noise(DepolarizingChannel(0.01))
    state = QuantumState.ghz_state(ind)
    observables = create_pauli_observables(n_qubits=ind)
    print(observables)
    for obs_idx, observable in enumerate(observables):
        # Get noisy expectation value (with noise model) - measure directly in observable's basis
        noisy_expectation = platform.compute_expectation_value(
            state, observable, shots=10000
        )

        # Get ideal expectation value (without noise) - compute analytically
        ideal_expectation = observable.expectation_value(state)
        #platform.get_ideal_expectation(
        #      state, observable, shots=10000)

        # Fingerprint is the deviation: F = E_noisy - E_ideal
        print("F = E_noisy - E_ideal", noisy_expectation - ideal_expectation, " = ", noisy_expectation, " - ", ideal_expectation)


>> OK!
==> Start.
Create 0:
QuantumState(name='|0>', n_qubits=1, state_vector=[1.+0.j 0.+0.j])
Counter({'0': 10})
Create 1:
QuantumState(name='|1>', n_qubits=1, state_vector=[0.+0.j 1.+0.j])
Counter({'1': 10})
=========================================== 1 Qubits ===========================
The state:  QuantumState(name='|1>', n_qubits=1, state_vector=[0.+0.j 1.+0.j])
==================== Idel:
[ideal] ⟨X⟩ = 0.0248
[ideal] ⟨Y⟩ = -0.0106
[ideal] ⟨Z⟩ = -1.0000
==================== Noisy
[Noisy] ⟨X⟩ = 0.0070
[Noisy] ⟨Y⟩ = 0.0052
[Noisy] ⟨Z⟩ = -1.0000
=========================================== 2 Qubits ===========================
The state:  QuantumState(name='|0+>', n_qubits=2, state_vector=[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j])
==================== Ideal:
[ideal] ⟨XX⟩ = -0.0022 or 0.0000 
[ideal] ⟨XY⟩ = 0.0160 or 0.0000 
[ideal] ⟨XZ⟩ = 0.0030 or 0.0000 
[ideal] ⟨YX⟩ = -0.0058 or 0.0000 
[ideal] ⟨YY⟩ = -0.0066 or 0.0000 
[ideal] ⟨YZ⟩ = -0.0004 or 0.0000 
[ideal] ⟨Z

# New section

In [ ]:
# Testing https://github.com/vili-1/SimSHADOW_artifact/blob/main/simshadow/platforms/cirq_platform.py

In [ ]:
"""
Cirq platform implementation for SimSHADOW quantum simulator validation.

Provides Cirq-specific quantum circuit execution and noise model integration.
"""

import numpy as np
import cirq
from typing import List, Dict, Optional, Tuple
from cirq import Simulator, DensityMatrixSimulator
from cirq.circuits import InsertStrategy


import sys
from pathlib import Path

print(">> OK!")
#PROJECT_ROOT = Path("/content/SimSHADOW_artifact-main/simshadow/")
#sys.path.insert(0, str(PROJECT_ROOT))

sys.path.append("/content/SimSHADOW_artifact-main/simshadow/")

from core.shadow_tomography import QuantumState, PauliObservable
from core.noise_models import NoiseChannel, DepolarizingChannel, AmplitudeDampingChannel, PhaseDampingChannel


class CirqPlatform:
    """
    Cirq simulator platform for SimSHADOW experiments.
    Handles circuit construction, noise model configuration, and measurement execution
    specifically for Cirq-based simulations.
    """

    def __init__(self, n_qubits: int = 2):
        self.platform_name = "Cirq"
        self.n_qubits = n_qubits
        self.qubits = cirq.LineQubit.range(n_qubits)
        self.simulator = DensityMatrixSimulator()  # Use density matrix for noise
        self.noise_config: Optional[Dict] = None
        self.current_noise_channel: Optional[NoiseChannel] = None

    def configure_noise(self, noise_channel: NoiseChannel):
        """
        Configure Cirq noise model based on SimSHADOW noise channel.
        Called from run_simshadow.py
        """
        self.current_noise_channel = noise_channel
        self.noise_config = {
            'type': noise_channel.name,
            'parameter': noise_channel.parameter
        }

    def _add_noise_to_circuit(self, circuit: cirq.Circuit) -> cirq.Circuit:
        """
        Add noise to circuit based on current noise configuration.
        Interal function only
        """
        if self.current_noise_channel is None:
            return circuit

        noisy_circuit = cirq.Circuit()

        for moment in circuit:
            # Add the original operations
            noisy_circuit.append(moment, strategy=InsertStrategy.NEW_THEN_INLINE)

            # Add noise after each gate
            for operation in moment:
                if isinstance(self.current_noise_channel, DepolarizingChannel):
                    # Add depolarizing noise to each qubit involved
                    for qubit in operation.qubits:
                        noise_op = cirq.depolarize(p=self.current_noise_channel.p).on(qubit)
                        noisy_circuit.append(noise_op, strategy=InsertStrategy.INLINE)

                elif isinstance(self.current_noise_channel, AmplitudeDampingChannel):
                    # Add amplitude damping noise
                    for qubit in operation.qubits:
                        noise_op = cirq.amplitude_damp(gamma=self.current_noise_channel.gamma).on(qubit)
                        noisy_circuit.append(noise_op, strategy=InsertStrategy.INLINE)

                elif isinstance(self.current_noise_channel, PhaseDampingChannel):
                    # Add phase damping noise
                    for qubit in operation.qubits:
                        noise_op = cirq.phase_damp(gamma=self.current_noise_channel.lambda_param).on(qubit)
                        noisy_circuit.append(noise_op, strategy=InsertStrategy.INLINE)
                else:
                    raise ValueError(
                        "NoiseModel.name must be valid. ",
                        "Use DepolarizingChannel, AmplitudeDampingChannel, or PhaseDampingChannel"
                    )

        return noisy_circuit

    def prepare_state_circuit(self, quantum_state: QuantumState) -> cirq.Circuit:
        """
        Create a Cirq circuit to prepare the specified quantum state.
        Internal function only.
        """
        # Here we will store the state!
        circuit = cirq.Circuit()

        if "GHZ" in quantum_state.name or "Φ" in quantum_state.name:
            # GHZ state preparation
            circuit.append(cirq.H(self.qubits[0]))
            for i in range(1, self.n_qubits):
                circuit.append(cirq.CNOT(self.qubits[0], self.qubits[i]))
        else: # We only have 0,1,+ and -

            # Explicitly keep only valid single-qubit symbols
            bitstring = [ch for ch in quantum_state.name if ch in {'0', '1', '+', '-'}]
            if len(bitstring) != self.n_qubits:
                raise ValueError(
                    f"State {quantum_state.name} does not match n_qubits={self.n_qubits}"
                )

            for i, ch in enumerate(bitstring):
                if ch == '0':
                    pass
                elif ch == '1':
                    circuit.append(cirq.X(self.qubits[i]))
                elif ch == '+':
                    circuit.append(cirq.H(self.qubits[i]))
                elif ch == '-':
                    circuit.append([cirq.H(self.qubits[i]), cirq.Z(self.qubits[i])])
                else:
                    raise ValueError(
                        "QuantumState.name must be valid. "
                        "Use 0, 1, +, - or GHZ only."
                    )
        return circuit

    def measure_pauli(self, quantum_state: QuantumState, pauli_string: str) -> str:
        """
        Measure quantum state in specified Pauli basis and return outcome.

        Args:
            quantum_state: The quantum state to measure
            pauli_string: Pauli string like 'XY', 'ZZ', etc.

        Returns:
            Measurement outcome as bitstring

        Called:
            measure_pauli from shadow_tomography.py
        """
        # Prepare initial state
        circuit = self.prepare_state_circuit(quantum_state)

        # Add Pauli basis rotations
        for i, pauli in enumerate(pauli_string):
            if pauli == 'X':
                circuit.append(cirq.ry(-np.pi/2)(self.qubits[i]))
            elif pauli == 'Y':
                circuit.append(cirq.rx(np.pi/2)(self.qubits[i]))
            # Leave this to the full paper, with a proper theorem.
            #elif pauli == 'I' or pauli == 'Z':
            #    circuit.append(cirq.I(self.qubits[i]))
            # Z measurement is in computational basis (no rotation needed)

        # Add noise if configured
        if self.current_noise_channel:
            circuit = self._add_noise_to_circuit(circuit)

        # Add measurements
        for i in range(self.n_qubits):
            circuit.append(cirq.measure(self.qubits[i], key=f'q{i}'))

        # Execute circuit
        result = self.simulator.run(circuit, repetitions=1)

        # Extract measurement outcome
        outcome_bits = []
        for i in range(self.n_qubits):
            outcome_bits.append(str(result.measurements[f'q{i}'][0][0]))
        #print(outcome_bits)

        return ''.join(outcome_bits)

    def compute_expectation_value(self,
                                quantum_state: QuantumState,
                                observable: PauliObservable,
                                shots: int = 10000) -> float:
        """
        Compute expectation value of Pauli observable with proper noise simulation.

        Args:
            quantum_state: The quantum state
            observable: Pauli observable to measure
            shots: Number of measurement shots

        Returns:
            Estimated expectation value

        Called:
            main from run_simshadow.py
        """
        # Prepare state circuit
        circuit = self.prepare_state_circuit(quantum_state)

        # Add Pauli basis rotations for measurement
        for i, pauli in enumerate(observable.pauli_string):
            if pauli == 'X':
                circuit.append(cirq.ry(-np.pi/2)(self.qubits[i]))
            elif pauli == 'Y':
                circuit.append(cirq.rx(np.pi/2)(self.qubits[i]))
            # Leave this to the full paper, with a proper theorem.
            #elif pauli == 'I' or pauli == 'Z':
            #    circuit.append(cirq.I(self.qubits[i]))

        # Add noise if configured
        if self.current_noise_channel:
            circuit = self._add_noise_to_circuit(circuit)

        # Add measurements
        for i in range(self.n_qubits):
            circuit.append(cirq.measure(self.qubits[i], key=f'q{i}'))

        # Execute circuit with multiple shots
        result = self.simulator.run(circuit, repetitions=shots)

        # Compute expectation value from measurement statistics
        # For Pauli observables, eigenvalue is product of individual qubit eigenvalues
        # After rotating to measurement basis: |0⟩ → +1, |1⟩ → -1
        # For identity (I): eigenvalue is always +1
        expectation = 0.0
        for shot in range(shots):
            # Get measurement outcome for this shot
            outcome_bits = []
            for i in range(self.n_qubits):
                outcome_bits.append(result.measurements[f'q{i}'][shot][0])

            # Compute eigenvalue as product of individual qubit eigenvalues
            eigenvalue = 1.0
            for i, pauli in enumerate(observable.pauli_string):
                if pauli == 'I':
                    # Identity always has eigenvalue +1
                    continue
                else:
                    # After rotation to Pauli basis, |0⟩ → +1, |1⟩ → -1
                    bit = outcome_bits[i]
                    qubit_eigenvalue = 1.0 if bit == 0 else -1.0
                    eigenvalue *= qubit_eigenvalue

            expectation += eigenvalue / shots

        return expectation

    ## WORKING VERSION OF IDEAL
    def get_ideal_expectation(self, quantum_state: QuantumState,
                              observable: PauliObservable,
                              shots: int = 10000) -> float:
        """Get ideal (noiseless) expectation value."""
        # Temporarily disable noise
        original_channel = self.current_noise_channel
        self.current_noise_channel = None

        expectation = self.compute_expectation_value(quantum_state, observable, shots)

        # Restore noise configuration
        self.current_noise_channel = original_channel

        return expectation

# So this works
print("Create 0:")
platform = CirqPlatform(1)
state = QuantumState.computational_basis_state("0")
print(state)
print(state.to_string())

samples = []
for _ in range(10):
    outcome = platform.measure_pauli(state, "Z")
    samples.append(outcome)
print(Counter(samples))

print("Create 1:")
platform = CirqPlatform(1)
state = QuantumState.computational_basis_state("1")
print(state)
print(state.to_string())

samples = []
for _ in range(10):
    outcome = platform.measure_pauli(state, "Z")
    samples.append(outcome)
print(Counter(samples))

# Set up platform with NO noise
print("=========================================== 1 Qubits ===========================")
platform = CirqPlatform(1)
state = QuantumState.computational_basis_state("1")
print("The state: ", state.to_string())
print("==================== Idel:")
for obs in ["X", "Y", "Z"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f}")
print("==================== Noisy")
for obs in ["X", "Y", "Z"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")

print("=========================================== 2 Qubits ===========================")
platform = CirqPlatform(2)
platform.configure_noise(DepolarizingChannel(0.01))

#state = state = QuantumState.ghz_state(2) #QuantumState.computational_basis_state("00")
superposition_patterns = ['+0', '-0', '0+', '0-']
pattern = superposition_patterns[2]
state = QuantumState.superposition_state(list(pattern))

#state = state = QuantumState.ghz_state(2) #QuantumState.computational_basis_state("00")
print("The state: ", state.to_string())
print("==================== Ideal:")
for obs in ["XX","XY","XZ","YX","YY","YZ","ZX","ZY","ZZ"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f}")

print("==================== Noisy")
for obs in ["XX","XY","XZ","YX","YY","YZ","ZX","ZY","ZZ"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")


print("=========================================== 3 Qubits ===========================")
platform = CirqPlatform(3)
platform.configure_noise(DepolarizingChannel(0.01))

state = QuantumState.ghz_state(3) #QuantumState.computational_basis_state("00")
print(state)
print("The state: ", state.to_string())
print("==================== Ideal:")
for obs in ["XXX","XXY","XXZ","XYX","XYY","XYZ","XZX","XZY","XZZ","YXX","YXY","YXZ","YYX","YYY","YYZ","YZX","YZY","YZZ","ZXX","ZXY","ZXZ","ZYX","ZYY","ZYZ","ZZX","ZZY","ZZZ"]:
    E = platform.get_ideal_expectation(state, PauliObservable(obs))
    print(f"[ideal] ⟨{obs}⟩ = {E:.4f}")

print("==================== Noisy")
for obs in ["XXX","XXY","XXZ","XYX","XYY","XYZ","XZX","XZY","XZZ","YXX","YXY","YXZ","YYX","YYY","YYZ","YZX","YZY","YZZ","ZXX","ZXY","ZXZ","ZYX","ZYY","ZYZ","ZZX","ZZY","ZZZ"]:
    E = platform.compute_expectation_value(state, PauliObservable(obs), shots=10000)
    print(f"[Noisy] ⟨{obs}⟩ = {E:.4f}")

for ind in [1, 2, 3]:
    print("=========================================== ", ind, " Qubits ===========================")
    platform = CirqPlatform(ind)
    platform.configure_noise(DepolarizingChannel(0.01))
    state = QuantumState.ghz_state(ind)
    observables = create_pauli_observables(n_qubits=ind)
    print(observables)
    for obs_idx, observable in enumerate(observables):
        # Get noisy expectation value (with noise model) - measure directly in observable's basis
        noisy_expectation = platform.compute_expectation_value(
            state, observable, shots=10000
        )

        # Get ideal expectation value (without noise) - compute analytically
        ideal_expectation = ideal_expectation = observable.expectation_value(state)
        #platform.get_ideal_expectation(
        #      state, observable, shots=10000)

        # Fingerprint is the deviation: F = E_noisy - E_ideal
        print("F = E_noisy - E_ideal", noisy_expectation - ideal_expectation, " = ", noisy_expectation, " - ", ideal_expectation)



>> OK!
Create 0:
QuantumState(name='|0>', n_qubits=1, state_vector=[1.+0.j 0.+0.j])
Counter({'0': 10})
Create 1:
QuantumState(name='|1>', n_qubits=1, state_vector=[0.+0.j 1.+0.j])
Counter({'1': 10})
=========================================== 1 Qubits ===========================
The state:  QuantumState(name='|1>', n_qubits=1, state_vector=[0.+0.j 1.+0.j])
==================== Idel:
[ideal] ⟨X⟩ = 0.0126
[ideal] ⟨Y⟩ = 0.0074
[ideal] ⟨Z⟩ = -1.0000
==================== Noisy
[Noisy] ⟨X⟩ = 0.0112
[Noisy] ⟨Y⟩ = 0.0068
[Noisy] ⟨Z⟩ = -1.0000
=========================================== 2 Qubits ===========================
The state:  QuantumState(name='|0+>', n_qubits=2, state_vector=[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j])
==================== Ideal:
[ideal] ⟨XX⟩ = -0.0082
[ideal] ⟨XY⟩ = -0.0116
[ideal] ⟨XZ⟩ = 0.0106
[ideal] ⟨YX⟩ = 0.0120
[ideal] ⟨YY⟩ = 0.0034
[ideal] ⟨YZ⟩ = 0.0064
[ideal] ⟨ZX⟩ = 1.0000
[ideal] ⟨ZY⟩ = -0.0012
[ideal] ⟨ZZ⟩ = 0.0102
==================== No